In [8]:
# system tools
import sys
from pathlib import Path

# battle processing
import json
from zipfile import ZipFile
from tools.battle import Battle
from tools.full_pokemon import FullPokemon
import copy

# data science
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score,StratifiedKFold,train_test_split
from sklearn.base import BaseEstimator,ClassifierMixin
from sklearn.metrics import accuracy_score
import statsmodels.api as sm

repo = Path.cwd().resolve()
sys.path.append(str(repo / "damage-calc-python-wrapper"))
sys.path.append(str(repo / "damage-calc-python-wrapper" / "python_calc"))

num_training_zips = 3 # change if you want fewer/more battles at the benefit/cost of less/more time; current max is 3
replay_dir = repo / "data" / "replays"
zip_paths = [replay_dir / f"gen9randombattles_{i}.zip" for i in range(1,num_training_zips+1)]
replay_zips = [ZipFile(zip_path,'r') for zip_path in zip_paths]

from python_calc import (
    Pokemon,
    Move,
    Field,
    Side,
    calc,
    advantage
)

In [9]:
class BaselineEloPredictor(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"
    coef_ = np.array([[np.log(10)/400]])

    def __init__(self, scale=400):
        self.scale = scale

    def fit(self, X, y,offset = None): #offset should not be used, it's merely there to have the same signature as log-reg-with-offset.fit
        X = np.asarray(X)
        y = np.asarray(y)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        return self

    def predict_proba(self, X,offset = None):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        elo_diff = X[:, 0].astype(float)
        p = 1 / (1 + 10 ** (-elo_diff / self.scale))
        return np.column_stack([1 - p, p])

    def predict(self, X, offset = None):
        probs = self.predict_proba(X)[:, 1]
        preds = (probs >= 0.5).astype(int)
        return self.classes_[preds]

    # def score(self, X, y):
    #     return np.mean(self.predict(X) == y)

    # def __sklearn_tags__(self):
    #     tags = super().__sklearn_tags__()
    #     tags.estimator_type = "classifier"
    #     return tags

In [11]:
class LogisticRegressionWithOffset(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def fit(self, X, y, offset=None):
        self.offset = offset
        X = np.asarray(X)
        y = np.asarray(y)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]


        self.fitted = sm.GLM(y, X, family=sm.families.Binomial(), offset=offset).fit()
        self.coef_ = np.array([self.fitted.params])
        return self

    def predict_proba(self, X,offset = None):
        p = np.array(self.fitted.predict(X, offset=offset)).reshape(-1, 1)
        return np.concatenate([1 - p, p], axis=1)

    def predict(self, X,offset = None):
        return 1*(self.predict_proba(X,offset=offset)[:,1]>=0.5)


## Example Calculations

In [4]:
palafin = Pokemon(
    name="Palafin-Hero",
    gen=9,
    level=100,
    moves=["Bulk Up", "Overheat","Wave Crash","Drain Punch"],
    evs={"hp" : 248, "atk" : 8, "spd" : 252},
    nature="Careful",
    item="Leftovers")

In [5]:
delphox = Pokemon(
    gen=9,
    name="Delphox",
    evs={"spa" : 252, "spe" : 252, "hp" : 4},
    nature = "Jolly",
    moves = ["fireblast"]
)

In [6]:
gengar = Pokemon(
    gen=9,
    name="Gengar",
    level=100,
    item="Choice Specs",
    nature="Timid",
    evs={"spa" : 252, "spe" : 252, "spd" : 4},
    moves = ["Shadow Ball", "Sludge Wave", "Thunderbolt", "Focus Blast"],
    curHP=261)

In [7]:
tauros_p_a = Pokemon(
    name="taurospaldeaaqua",
    gen=9,
    level=100
)

In [8]:
fire_blast = Move(gen=9,name="fireblast")

In [9]:
calc.calculate(gen = 9, attacker = delphox, defender = tauros_p_a, move = fire_blast,field=Field())

{'gen': 9,
 'attacker': {'name': 'Delphox',
  'ability': 'Blaze',
  'item': None,
  'level': 100,
  'nature': 'Jolly',
  'types': ['Fire', 'Psychic'],
  'stats': {'hp': 292,
   'atk': 174,
   'def': 180,
   'spa': 294,
   'spd': 236,
   'spe': 337},
  'rawStats': {'hp': 292,
   'atk': 174,
   'def': 180,
   'spa': 294,
   'spd': 236,
   'spe': 337},
  'boosts': {'hp': 0, 'atk': 0, 'def': 0, 'spa': 0, 'spd': 0, 'spe': 0},
  'originalCurHP': 292,
  'isDynamaxed': None,
  'teraType': None,
  'status': '',
  'toxicCounter': 0},
 'defender': {'name': 'taurospaldeaaqua',
  'ability': 'Intimidate',
  'item': None,
  'level': 100,
  'nature': 'Serious',
  'types': ['Fighting', 'Water'],
  'stats': {'hp': 291,
   'atk': 256,
   'def': 246,
   'spa': 96,
   'spd': 176,
   'spe': 236},
  'rawStats': {'hp': 291,
   'atk': 256,
   'def': 246,
   'spa': 96,
   'spd': 176,
   'spe': 236},
  'boosts': {'hp': 0, 'atk': 0, 'def': 0, 'spa': 0, 'spd': 0, 'spe': 0},
  'originalCurHP': 291,
  'isDynamaxed':

In [10]:
id = "2631360263"
with open("data/replays/gen9-randombattle/gen9randombattle-" + id + ".json") as battle_json:
    data = json.load(battle_json)

team1 = [
    Pokemon(
        name=data["teams_full"][0][mon_name]["speciesId"],
        gen=9,
        level=data["teams_full"][0][mon_name]["level"],
        ability = data["teams_full"][0][mon_name]["ability"],
        item = data["teams_full"][0][mon_name]["item"],
        gender = data["teams_full"][0][mon_name]["gender"],
        ivs = data["teams_full"][0][mon_name]["ivs"],
        evs = data["teams_full"][0][mon_name]["evs"],
        # teraType = data["teams_full"][0][mon_name]["teraType"],
        moves = data["teams_full"][0][mon_name]["moves"]
    )
    for mon_name in data["teams_full"][0].keys()
]

team2 = [
    Pokemon(
        name=data["teams_full"][1][mon_name]["speciesId"],
        gen=9,
        level=data["teams_full"][1][mon_name]["level"],
        ability = data["teams_full"][1][mon_name]["ability"],
        item = data["teams_full"][1][mon_name]["item"],
        gender = data["teams_full"][1][mon_name]["gender"],
        ivs = data["teams_full"][1][mon_name]["ivs"],
        evs = data["teams_full"][1][mon_name]["evs"],
        # teraType = data["teams_full"][1][mon_name]["teraType"], # calculate will assume that the teraType is on
        moves = data["teams_full"][1][mon_name]["moves"]
    )
    for mon_name in data["teams_full"][1].keys()
]

In [11]:
rows = [[team1[i].name] + [advantage(m1=team1[i],m2=team2[j]) for j in range(6)] for i in range(6)]
df = pd.DataFrame(rows,columns=['team1'] + [team2[j].name for j in range(6)])
df

,team1,wigglytuff,lokix,mandibuzz,snorlax,torterra,mienshao
0,delphox,1.041038,2.244024,0.610622,0.592408,1.572011,1.573258
1,indeedeef,0.695286,0.635868,0.333409,-0.212124,0.607267,0.654736
2,skarmory,-1.267756,1.192458,-0.262036,0.755850,1.204255,0.636454
3,wugtrio,0.543337,0.718043,-0.619853,0.426934,0.104628,0.783787
4,taurospaldeaaqua,0.320318,1.540621,0.954355,1.254408,0.764140,0.525421
5,victreebel,1.212396,0.854022,-0.529205,0.922409,0.887759,0.772303


## Comparing new and old

In [4]:
files = [replay_zip.read(file_name) for replay_zip in replay_zips for file_name in replay_zip.namelist()]
rows = []

for file in files:
    try:
        data = json.loads(file)

        # new advantage calcs
        team1 = [
            Pokemon(
                name=data["teams_full"][0][mon_name]["speciesId"],
                gen=9,
                level=data["teams_full"][0][mon_name]["level"],
                ability = data["teams_full"][0][mon_name]["ability"],
                item = data["teams_full"][0][mon_name]["item"],
                gender = data["teams_full"][0][mon_name]["gender"],
                ivs = data["teams_full"][0][mon_name]["ivs"],
                evs = data["teams_full"][0][mon_name]["evs"],
                # teraType = data["teams_full"][0][mon_name]["teraType"],
                moves = data["teams_full"][0][mon_name]["moves"]
            )
            for mon_name in data["teams_full"][0].keys()
        ]

        team2 = [
            Pokemon(
                name=data["teams_full"][1][mon_name]["speciesId"],
                gen=9,
                level=data["teams_full"][1][mon_name]["level"],
                ability = data["teams_full"][1][mon_name]["ability"],
                item = data["teams_full"][1][mon_name]["item"],
                gender = data["teams_full"][1][mon_name]["gender"],
                ivs = data["teams_full"][1][mon_name]["ivs"],
                evs = data["teams_full"][1][mon_name]["evs"],
                # teraType = data["teams_full"][1][mon_name]["teraType"], # calculate will assume that the teraType is on
                moves = data["teams_full"][1][mon_name]["moves"]
            )
            for mon_name in data["teams_full"][1].keys()
        ]

        new_advs = [advantage(gen=9,m1=team1[i],m2=team2[j]) for i in range(len(team1)) for j in range(len(team2))]



        # old advantage calcs
        battle = Battle(data_json=data,parse=True)
        team1 = [FullPokemon(battle.teams_full[0][mon]) for mon in battle.teams_full[0].keys()]
        team2 = [FullPokemon(battle.teams_full[1][mon]) for mon in battle.teams_full[1].keys()]

        
        if not battle.custom_ruleQ:
            rows.append({
                    "id": battle.id,
                    "p1": battle.players[0],
                    "p2": battle.players[1],
                    "duration": battle.end_time - battle.start_time,
                    "p1_rating" : battle.player_dets[0]["rating"],
                    "elo_diff": battle.player_dets[0]["rating"] - battle.player_dets[1]["rating"],
                    "p1_wins" : battle.players[0] == battle.winner.name,
                    "p1_revealed_team_size" : len(battle.teams[0].keys()),
                    "p2_revealed_team_size" : len(battle.teams[1].keys()),
                    "new_adv" : sum(new_advs),
                    "old_adv" : sum(FullPokemon.advantage(team1[m1],team2[m2]) for m1 in range(6) for m2 in range(6))
                })
    except (json.JSONDecodeError,UnicodeDecodeError):
        continue

full_match_data = pd.DataFrame(rows)

elo_diff_coef = np.log(10) / 400
full_match_data["elo_diff_offset"] = elo_diff_coef * full_match_data["elo_diff"]

In [90]:
# We should throw away matches where people rage quit early
complete_matches = full_match_data[(full_match_data['duration'] > 60) & ((full_match_data["p1_revealed_team_size"] > 2) | (full_match_data["p2_revealed_team_size"] > 2))]
# This is to grab matches where we know that the players understand the basic switching strategy (from Marz' work on switching), 1965 is standard
threshold = 1965
highly_rated_matches = complete_matches[(complete_matches['p1_rating'] > threshold) & (complete_matches[['p1_rating','elo_diff']].sum(axis=1) > threshold)]

In [91]:
highly_rated_matches

,id,p1,p2,duration,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,new_adv,old_adv,elo_diff_offset
3,gen9randombattle-2631529004,WhatEver2102,Duck Cop,123,1999,17,True,1,3,9.719376,6.441962,0.097860
4,gen9randombattle-2631993792,monomythic,OverthereStair,301,2120,58,False,6,6,-1.283986,1.231337,0.333875
7,gen9randombattle-2631439736,Mr Brightside,indias last hope,448,2115,-144,False,6,6,10.275611,6.048035,-0.828931
8,gen9randombattle-2631771408,Illuminating_Fate,medo6037,287,2170,-7,True,5,4,-4.767283,4.106191,-0.040295
14,gen9randombattle-2631594339,szbsb,Bigoleg,417,2047,-36,True,5,6,5.587282,1.795251,-0.207233
...,...,...,...,...,...,...,...,...,...,...,...,...
12749,gen9randombattle-2642114009,forcemajor14,majex,196,2084,2,False,6,6,2.770689,-0.835965,0.011513
12756,gen9randombattle-2642143701,forcemajor14,lexam22,246,2235,-12,False,6,1,5.940640,0.514204,-0.069078
12759,gen9randombattle-2641938445,andonibavi,qiuescent,277,2173,-43,True,5,5,-8.897955,-2.654992,-0.247528
12763,gen9randombattle-2642095210,notmetbh102,Uday30,187,2220,-49,False,6,5,7.833947,6.974347,-0.282067


In [100]:
# Let's check the p-values of each individual feature in a logistic regression
features = ['new_adv', 'old_adv']
df = complete_matches # good options include: complete_matches, highly_rated_matches

models = [sm.Logit(df['p1_wins'],df[[feature]]).fit(disp=False) for feature in features]

rows = []
for i in range(len(features)):
    summary = pd.Series({"feature" : features[i], "p-value" : models[i].pvalues.iloc[0], "coefficient" : models[i].params.iloc[0]})
    rows.append(summary)

table = pd.DataFrame(rows)
table.sort_values("p-value",ascending=True)

,feature,p-value,coefficient
0,new_adv,9.832785e-13,0.014962
1,old_adv,1.465210e-10,0.018167


In [106]:
n_splits = 10
lr = LogisticRegression(C=np.inf,fit_intercept=False,random_state=207,max_iter=1000)
skf = StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=207)

model_info = [
    ('new_adv_only',LogisticRegressionWithOffset(),df[["new_adv"]],None),
    ('old_adv_only',LogisticRegressionWithOffset(),df[["old_adv"]],None),
    ('elo_diff_only',BaselineEloPredictor(),df[["elo_diff"]],None),
    ('new_adv_elo_diff',LogisticRegressionWithOffset(),df[["new_adv"]],df["elo_diff_offset"]),
    ('old_adv_elo_diff',LogisticRegressionWithOffset(),df[["old_adv"]],df["elo_diff_offset"])
]

model_accs = np.zeros(shape=(len(model_info),n_splits))

for model_index,(model_name,model,data_set,offset) in enumerate(model_info):
    for fold_index,(train_indices,test_indices) in enumerate(skf.split(X = df,y=df["p1_wins"])):
        # set up train-train sets
        X_tt = data_set.iloc[train_indices]
        y_tt = df.iloc[train_indices]["p1_wins"]
        if isinstance(offset,pd.Series):
            offset_tt = offset.iloc[train_indices]
        else:
            offset_tt = None

        # set up validation sets
        X_val = data_set.iloc[test_indices]
        y_val = df.iloc[test_indices]["p1_wins"]
        if isinstance(offset,pd.Series):
            offset_val = offset.iloc[test_indices]
        else:
            offset_val = None

        # fit and predict
        model.fit(X=X_tt,y=y_tt,offset = offset_tt)
        preds = model.predict(X=X_val,offset=offset_val)
        model_accs[model_index,fold_index] = accuracy_score(y_true=y_val,y_pred=preds)
    cv_scores = model_accs[model_index,:]
    print(f"The average accuracy score for the {model_name} model is {100*np.mean(cv_scores):.2f} +/- {100*np.std(cv_scores,ddof=1):.2f}%.")
    print(f"On the last fold, the coefficients for this model were {model.coef_[0][0]:.4f}.")
    print()
    


# for model_name,model,data_set,offset in model_info:
#     cv_scores = cross_val_score(estimator=model,X=data_set,y=df['p1_wins'],cv=skf,n_jobs=-1,scoring="accuracy")
#     print(f"The average accuracy score for the {model_name} model is {np.mean(cv_scores):.4f} +/- {np.std(cv_scores,ddof=1):.4f}.")
#     model.fit(X=data_set,y=df["p1_wins"])
#     print(f"On the entire training set, the coefficients for this model are {model.coef_}.")
#     print()

The average accuracy score for the new_adv_only model is 52.73 +/- 1.26%.
On the last fold, the coefficients for this model were 0.0152.

The average accuracy score for the old_adv_only model is 52.39 +/- 0.73%.
On the last fold, the coefficients for this model were 0.0181.

The average accuracy score for the elo_diff_only model is 52.17 +/- 0.91%.
On the last fold, the coefficients for this model were 0.0058.

The average accuracy score for the new_adv_elo_diff model is 53.06 +/- 0.96%.
On the last fold, the coefficients for this model were 0.0159.

The average accuracy score for the old_adv_elo_diff model is 53.01 +/- 1.14%.
On the last fold, the coefficients for this model were 0.0185.



In [15]:
# look at calibration for new v old
# look at confusion matrices for new v old
# look into calibration in-the-large
# look into range of predict_proba outcomes